## LSTM: Sentimental Analysis On Toy Datasets:

Requires concept of:
- Embeding
- LSTM
- Tokenization and Vocab construction

## 1. Imports

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re

## 2. Dataset Loader

In [2]:
# 1. Dataset class and preprocessing
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = [self.vocab.get(token, self.vocab['<unk>']) for token in self.texts[idx].split()]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

## 3. Tokenization And Vocabulary Building:

In [3]:
# 2. Simple tokenizer and vocabulary builder
def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

def build_vocab(texts, min_freq=1):
    freq = {}
    for text in texts:
        for token in simple_tokenizer(text):
            freq[token] = freq.get(token, 0) + 1
    vocab = {'<pad>':0, '<unk>':1}
    idx = 2
    for word, count in freq.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

## 4. LSTM Module:

In [4]:
# 3. Model definition
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        out = self.fc(hidden[-1])
        return out

## 5. Exploring Toy Dataset:

In [21]:
# Example dataset (replace with your dataset or load real data)
texts = [
    "I loved the movie it was fantastic",
    "The film was terrible and boring",
    "Great acting and wonderful story",
    "I did not like the movie",
    "The movie was okay not great",
    "Absolutely fantastic and enjoyable",
    "Worst movie I have seen",
    "It was an average film",
]
labels = [1, 0, 1, 0, 1, 1, 0, 1]  # 1 = positive, 0 = negative

# Build vocab
vocab = build_vocab(texts)

# Train-validation split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.25, random_state=42)

# Create datasets and loaders
train_dataset = SentimentDataset(train_texts, train_labels, vocab)
val_dataset = SentimentDataset(val_texts, val_labels, vocab)

## 6. Load Datasets As Tensor:

**Collate**

In [16]:
def collate_fn(batch):
    """
    Comginess individual samples into a batch by padding variable-length sequences so they all 
    have the same length, allowing the model to process them together efficiently.
    """
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels


# Example sequences (tensors) with different lengths
seq1 = torch.tensor([1, 2, 3])
seq2 = torch.tensor([4, 5])
seq3 = torch.tensor([6])

batch = [seq1, seq2, seq3]

# Using pad_sequence to pad to the max length (3 here)
padded_batch = pad_sequence(batch, batch_first=True, padding_value=0)

print(padded_batch)

tensor([[1, 2, 3],
        [4, 5, 0],
        [6, 0, 0]])


In [14]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

# Model, loss, optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentimentLSTM(len(vocab), embedding_dim=50, hidden_dim=64, output_dim=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

**Training and Evaluation**

In [10]:
# Training and validation loop
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc

# Run training
num_epochs = 10
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


Epoch 1/10 - Train Loss: 0.6965, Train Acc: 0.5000 - Val Loss: 0.6900, Val Acc: 0.5000
Epoch 2/10 - Train Loss: 0.6684, Train Acc: 0.5000 - Val Loss: 0.6861, Val Acc: 0.5000
Epoch 3/10 - Train Loss: 0.6365, Train Acc: 0.6667 - Val Loss: 0.6829, Val Acc: 0.5000
Epoch 4/10 - Train Loss: 0.6049, Train Acc: 0.8333 - Val Loss: 0.6805, Val Acc: 0.5000
Epoch 5/10 - Train Loss: 0.5709, Train Acc: 0.8333 - Val Loss: 0.6782, Val Acc: 0.5000
Epoch 6/10 - Train Loss: 0.5359, Train Acc: 1.0000 - Val Loss: 0.6766, Val Acc: 0.5000
Epoch 7/10 - Train Loss: 0.5072, Train Acc: 1.0000 - Val Loss: 0.6751, Val Acc: 0.5000
Epoch 8/10 - Train Loss: 0.4770, Train Acc: 1.0000 - Val Loss: 0.6749, Val Acc: 0.5000
Epoch 9/10 - Train Loss: 0.4326, Train Acc: 1.0000 - Val Loss: 0.6762, Val Acc: 0.5000
Epoch 10/10 - Train Loss: 0.3968, Train Acc: 1.0000 - Val Loss: 0.6806, Val Acc: 0.5000


**Inference**

In [8]:
def predict_sentiment(model, sentence, vocab, device):
    model.eval()
    # Preprocess sentence: tokenize, lowercase, remove punctuation
    import re
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z0-9\s]', '', sentence)
    tokens = sentence.split()
    
    # Convert tokens to indices using vocab (use <unk> if missing)
    indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
    
    # Convert to tensor and add batch dimension
    input_tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred_class].item()
    
    sentiment = "Positive" if pred_class == 1 else "Negative"
    return sentiment, confidence

# Example usage after training:
sentence = "The movie was fantastic and thrilling"
sentiment, confidence = predict_sentiment(model, sentence, vocab, device)
print(f"Sentence: {sentence}\nPredicted sentiment: {sentiment} (Confidence: {confidence:.2f})")


Sentence: The movie was fantastic and thrilling
Predicted sentiment: Positive (Confidence: 0.64)
